In [1]:
#1
d <- read.csv("Loan.csv", header = TRUE)
d$Education <- as.factor(d$Education)
summary(d)

      Loan           Income           Family          CCAvg        Education
 Min.   :0.000   Min.   :  8.00   Min.   :1.000   Min.   : 0.000   1:2096   
 1st Qu.:0.000   1st Qu.: 39.00   1st Qu.:1.000   1st Qu.: 0.700   2:1403   
 Median :0.000   Median : 64.00   Median :2.000   Median : 1.500   3:1501   
 Mean   :0.096   Mean   : 73.77   Mean   :2.396   Mean   : 1.938            
 3rd Qu.:0.000   3rd Qu.: 98.00   3rd Qu.:3.000   3rd Qu.: 2.500            
 Max.   :1.000   Max.   :224.00   Max.   :4.000   Max.   :10.000            

In [4]:
#2,3
model1 <- lm(Loan ~ Income + Family + CCAvg + Education, data = d)
summary(model1)


Call:
lm(formula = Loan ~ Income + Family + CCAvg + Education, data = d)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.56354 -0.14730 -0.03822  0.06978  1.05386 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -3.455e-01  1.127e-02 -30.653  < 2e-16 ***
Income       3.367e-03  9.799e-05  34.364  < 2e-16 ***
Family       3.160e-02  3.010e-03  10.499  < 2e-16 ***
CCAvg        1.373e-02  2.538e-03   5.412 6.52e-08 ***
Education2   1.517e-01  8.473e-03  17.908  < 2e-16 ***
Education3   1.605e-01  8.229e-03  19.511  < 2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.2393 on 4994 degrees of freedom
Multiple R-squared:  0.3409,	Adjusted R-squared:  0.3402 
F-statistic: 516.6 on 5 and 4994 DF,  p-value: < 2.2e-16


In [12]:
min(model1$fitted.values)


[1] -0.2856269

In [6]:
#4
model2 <- glm(Loan ~ Income + Family + CCAvg + Education, data = d, family = binomial(link = "logit"))
summary(model2)


Call:
glm(formula = Loan ~ Income + Family + CCAvg + Education, family = binomial(link = "logit"), 
    data = d)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)    
(Intercept) -13.177833   0.517777 -25.451  < 2e-16 ***
Income        0.059791   0.002687  22.255  < 2e-16 ***
Family        0.587079   0.071275   8.237  < 2e-16 ***
CCAvg         0.162679   0.040505   4.016 5.91e-05 ***
Education2    3.910609   0.251037  15.578  < 2e-16 ***
Education3    3.933173   0.244329  16.098  < 2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 3162.0  on 4999  degrees of freedom
Residual deviance: 1334.8  on 4994  degrees of freedom
AIC: 1346.8

Number of Fisher Scoring iterations: 8


In [8]:
#5
threshold <- mean(d$Loan)
# Get predicted probabilities
pred_probs <- predict(model2, type = "response")

# Classify based on the threshold
pred_classes <- ifelse(pred_probs > threshold, 1, 0)
conf_matrix <- table(Actual = d$Loan, Predicted = pred_classes)
print(conf_matrix)

# Extract matrix elements safely
TN <- conf_matrix[1, 1] # Actual 0, Predicted 0
FN <- conf_matrix[2, 1] # Actual 1, Predicted 0
FP <- conf_matrix[1, 2] # Actual 0, Predicted 1
TP <- conf_matrix[2, 2] # Actual 1, Predicted 1

# 1. Overall PCP
pcp_overall <- (TP + TN) / sum(conf_matrix) * 100

# 2. PCP for y = 0 (Specificity / Negative Correct Rate)
pcp_y0 <- TN / (TN + FP) * 100

# 3. PCP for y = 1 (Sensitivity / Positive Correct Rate)
pcp_y1 <- TP / (TP + FN) * 100

# Print results cleanly
cat(sprintf("Overall PCP: %.2f%%\n", pcp_overall))
cat(sprintf("PCP for y = 0: %.2f%%\n", pcp_y0))
cat(sprintf("PCP for y = 1: %.2f%%\n", pcp_y1))

      Predicted
Actual    0    1
     0 4001  519
     1   61  419
Overall PCP: 88.40%
PCP for y = 0: 88.52%
PCP for y = 1: 87.29%


In [11]:
# 1. Grab the predicted probability (p) from Part 6
# Create the mean values profile for prediction
mean_values <- data.frame(
  Income    = mean(d$Income, na.rm = TRUE),
  Family    = mean(d$Family, na.rm = TRUE),
  CCAvg     = mean(d$CCAvg, na.rm = TRUE),
  Education = "2"
)

# Preview the profile
print(mean_values)
# (Assuming you created a data frame 'mean_values' containing the requested profile)
p <- predict(model2, newdata = mean_values, type = "response")

# 2. Extract Logit coefficients (excluding intercept)
logit_coefs <- coef(model2)[-1]

# 3. Calculate Logit Partial Effects
# Note: For strict partial effects of dummy variables (Education), a finite difference 
# (P(Ed=2) - P(Ed=1)) is technically preferred, but multiplying by p(1-p) is the standard continuous approximation.
logit_partial_effects <- logit_coefs * p * (1 - p)

# 4. Extract LPM coefficients (these ARE the partial effects for LPM)
# (Assuming your linear model is named 'model1')
lpm_partial_effects <- coef(model1)[-1] 

# 5. Co-list them in a data frame to compare
comparison_table <- data.frame(
  Variable = names(logit_coefs),
  LPM_Partial_Effect = lpm_partial_effects,
  Logit_Partial_Effect = logit_partial_effects
)
print(comparison_table)

   Income Family    CCAvg Education
1 73.7742 2.3964 1.937938         2
             Variable LPM_Partial_Effect Logit_Partial_Effect
Income         Income        0.003367256          0.002390594
Family         Family        0.031602457          0.023472981
CCAvg           CCAvg        0.013734629          0.006504346
Education2 Education2        0.151743228          0.156356601
Education3 Education3        0.160546743          0.157258760
